# 05 · Outbreak investigation: counts, rates, uncertainty

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-WorldView/blob/8e8feea094d489192b8a101069d32fa1328c764f/content/05_spatial_epidemiology.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/05_spatial_epidemiology.ipynb` at `8e8feea094d489192b8a101069d32fa1328c764f`; original SHA-256 `9e98aa59fa9162f6b93068bfb1f224a1c7a8ac402ffbbd23370298753f6e511c`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 05 · Outbreak investigation: counts, rates, uncertainty

**Mission:** build an epidemic curve and choropleth, compare denominators, and test reporting assumptions. Allow 45 minutes. Prerequisite: Lab 00.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then Run → Run All Cells. First use downloads Matplotlib. All training data are **SYNTHETIC** unless explicitly labeled LIVE. Downloads appear below and in `exports/`; the browser filesystem is separate from your computer. Each lab runs independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("matplotlib")
import json, math
from pathlib import Path
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from worldview_lab import (load, export, map_layer, chart_style, haversine_km,
    wilson_interval, shortest_times, freshness, normalize_quakes,
    validate_geojson, brief_request, validate_brief)
chart_style()

## Define the quantity before mapping it

The fixture contains 14-day **new cases**, with one case per fictional person, and a fixed resident population. `cases / population × 100,000` is a scaled **cumulative incidence proportion** for this interval. The short column name `rate_per_100k` is retained for map interoperability; this is not a person-time incidence rate. No claim about a real outbreak is made.

In [ ]:
districts=load('districts.geojson');p=[f['properties'] for f in districts['features']]
daily=load('daily_cases.json')
days=sorted({r['onset_date'] for r in daily})
counts=[sum(r['cases'] for r in daily if r['onset_date']==day) for day in days]
assert sum(counts)==sum(r['cases'] for r in p)
fig,ax=plt.subplots();ax.bar(range(1,15),counts);ax.set(xlabel='September 2026 onset day',ylabel='New cases',title='SYNTHETIC epidemic curve — onset date, 14 days');plt.show()

In [ ]:
rates=[r['cases']/r['population']*100000 for r in p]
intervals=[wilson_interval(r['cases'],r['population']) for r in p]
fig,ax=plt.subplots()
ax.errorbar(rates,[r['id'] for r in p],xerr=[[v-lo for v,(lo,hi) in zip(rates,intervals)], [hi-v for v,(lo,hi) in zip(rates,intervals)]],fmt='o',capsize=3)
ax.set(xlabel='14-day new cases per 100,000 residents',title='SYNTHETIC proportions with illustrative Wilson 95% intervals');plt.show()
map_layer(districts)

The intervals assume independent binomial outcomes and a complete denominator. They do **not** capture spatial dependence, under-ascertainment, reporting delays, or errors in population estimates. These teaching assumptions are usually incomplete in surveillance data. A cluster of colors is not a statistically established hotspot or a causal mechanism.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].bar([r['id'] for r in p],[r['cases'] for r in p]);axs[0].set(title='Case counts',ylabel='14-day cases')
adjusted=[v/r['reporting_fraction'] for v,r in zip(rates,p)]
axs[1].plot([r['id'] for r in p],rates,'o-',label='Observed exercise counts')
axs[1].plot([r['id'] for r in p],adjusted,'s--',label='Assumed under-reporting adjustment')
axs[1].set(title='Sensitivity, not an estimate',ylabel='Cases per 100,000');axs[1].legend(fontsize=8)
for ax in axs:ax.tick_params(axis='x',rotation=60)
fig.tight_layout();plt.show()
export('outbreak.geojson',districts)

## Lab challenge

Change one district denominator by ±20%. Compare count ranking, normalized ranking, and interval width. Explain why report-date and onset-date curves answer different questions. **Checkpoint:** D07 has the most cases and the highest scaled proportion in this fixture, but D05 exceeds several higher-count districts after normalization.

**Astra prompt:** “Use only the attached chart and definitions. Explain counts versus cumulative incidence in plain language, identify denominator and reporting biases, and propose verification questions. Avoid causal or individual-level inference.”

## Sources and attribution

[CDC: Describing epidemiologic data](https://www.cdc.gov/field-epi-manual/php/chapters/describing-epi-data.html) covers time, place, person, and visual interpretation. [NIST: binomial proportion intervals](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm) describes Wilson intervals.

World View inspiration: [Kevin (Khoa) To, MIT](https://github.com/kevtoe/worldview). Original lab code, figures, and synthetic fixtures: JupyterLite WorldView contributors, MIT. See the [book references](https://jltobias.github.io/JupyterLite-WorldView/book/references.html) and repository notices for dependency and service terms.